In [64]:
import pandas as pd 
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [65]:
df = pd.read_csv('../data/dataset.csv')

df.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Source
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0,Real
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0,Real
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0,Real
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0,Real
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0,Real


In [66]:
df_clean = df.copy()
df_clean['Date'] = pd.to_datetime(df_clean['Date'], format='%d/%m/%Y')
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month
df_clean.drop('Date', axis=1, inplace=True)
df_clean['Car'].head(30)

0     1.0
1     0.0
2     0.0
3     1.0
4     2.0
5     0.0
6     0.0
7     2.0
8     1.0
9     2.0
10    1.0
11    2.0
12    1.0
13    1.0
14    1.0
15    2.0
16    1.0
17    1.0
18    0.0
19    1.0
20    2.0
21    2.0
22    2.0
23    1.0
24    1.0
25    0.0
26    1.0
27    1.0
28    1.0
29    2.0
Name: Car, dtype: float64

In [67]:
# 2. MISSING VALUES (Imputation)
# Fill holes in BuildingArea/YearBuilt with Median
cols_with_nan = ['BuildingArea', 'YearBuilt', 'Car']
df_clean[cols_with_nan] = df_clean[cols_with_nan].fillna(df_clean[cols_with_nan].median())
print(f"✅ Missing values filled in {cols_with_nan}.")

✅ Missing values filled in ['BuildingArea', 'YearBuilt', 'Car'].


In [68]:
# 3. CATEGORICAL ENCODING
# Text -> Numbers (e.g., 'Region_1' -> 1)
categories_cols = ['Type', 'Method', 'Regionname', 'CouncilArea']
for col in categories_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
print(f"✅ Categorical columns encoded: {categories_cols}.")
df_clean.head()

✅ Categorical columns encoded: ['Type', 'Method', 'Regionname', 'CouncilArea'].


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Distance,Postcode,Bedroom2,...,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Source,Year,Month
0,Abbotsford,85 Turner St,2,0,1480000.0,1,Biggin,2.5,3067.0,2.0,...,135.921593,1966.0,39,-37.7996,144.9984,2,4019.0,Real,2016,12
1,Abbotsford,25 Bloomburg St,2,0,1035000.0,1,Biggin,2.5,3067.0,2.0,...,79.000000,1900.0,39,-37.8079,144.9934,2,4019.0,Real,2016,2
2,Abbotsford,5 Charles St,3,0,1465000.0,3,Biggin,2.5,3067.0,3.0,...,150.000000,1900.0,39,-37.8093,144.9944,2,4019.0,Real,2017,3
3,Abbotsford,40 Federation La,3,0,850000.0,0,Biggin,2.5,3067.0,3.0,...,135.921593,1966.0,39,-37.7969,144.9969,2,4019.0,Real,2017,3
4,Abbotsford,55a Park St,4,0,1600000.0,4,Nelson,2.5,3067.0,3.0,...,142.000000,2014.0,39,-37.8072,144.9941,2,4019.0,Real,2016,6


In [69]:

# 4. DROPPING USELESS COLUMNS
# Address is too unique; Suburb is covered by Postcode/Lat/Long
drop_cols = ['Address', 'SellerG', 'Suburb']
df_clean.drop(columns=drop_cols, inplace=True)

# 5. FEATURE SCALING
# Scale everything except Targets (Price, Tier) and Source
features_to_scale = [c for c in df_clean.columns if c not in ['Price','Price_Tier', 'Source' ]]
scaler = StandardScaler()
df_clean[features_to_scale] = scaler.fit_transform(df_clean[features_to_scale])
print("✅ Features scaled (Mean=0, Std=1).")


✅ Features scaled (Mean=0, Std=1).


In [70]:
p33 = np.percentile(df_clean['Price'], 33)
p66 = np.percentile(df_clean['Price'], 66)

def price_tier(price):
    if price < p33: return 'Low'
    elif price < p66: return 'Mid'
    else: return 'High'
    
# 3. Create the Column
df_clean['Price_Tier'] = df_clean['Price'].apply(price_tier)

In [72]:

df_final = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)
df_final.head()

df_final.to_csv('../data/final_melb_housing_data.csv', index=False)

print("✅ File saved as 'final_data.csv'")
print(f"Columns included: {list(df_final.columns)}")

df_clean.head()


✅ File saved as 'final_data.csv'
Columns included: ['Rooms', 'Type', 'Price', 'Method', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude', 'Longtitude', 'Regionname', 'Propertycount', 'Source', 'Year', 'Month', 'Price_Tier']


,Rooms,Type,Price,Method,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,...,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Source,Year,Month,Price_Tier
0,-0.868860,-0.763027,1480000.0,-0.442074,-1.331288,-0.121565,-0.848186,-0.871638,-0.463327,-0.092043,...,0.079850,1.525908,0.238498,-0.086277,-1.13193,-0.501438,Real,-0.656517,1.815573,High
1,-0.868860,-0.763027,1035000.0,-0.442074,-1.331288,-0.121565,-0.848186,-0.871638,-1.504400,-0.105522,...,-2.082365,1.525908,0.213774,-0.100305,-1.13193,-0.501438,Real,-0.656517,-1.733662,Mid
2,0.038481,-0.763027,1465000.0,1.111352,-1.331288,-0.121565,0.053582,0.452676,-1.504400,-0.111968,...,-2.082365,1.525908,0.209604,-0.097500,-1.13193,-0.501438,Real,-0.502113,-1.378738,High
3,0.038481,-0.763027,850000.0,-1.218788,-1.331288,-0.121565,0.053582,0.452676,-0.463327,-0.123689,...,0.079850,1.525908,0.246540,-0.090486,-1.13193,-0.501438,Real,-0.502113,-1.378738,Mid
4,0.945823,-0.763027,1600000.0,1.888065,-1.331288,-0.121565,0.053582,-0.871638,0.577745,-0.116071,...,1.652370,1.525908,0.215859,-0.098341,-1.13193,-0.501438,Real,-0.656517,-0.313968,High
